Evaluating Instruction Responses Locally Using a Llama 3 Model Via Ollama

In [ ]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar（进度条）
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import json
import requests


# 通过本地 Ollama API 调用 llama3 作为「裁判」模型（需先 `ollama pull llama3`）
def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {     # Settings below are required for deterministic responses（下面三项保证评分可复现）
            "seed": 123,        # 固定随机种子
            "temperature": 0,   # 温度 0 = 贪心、无随机
            "num_ctx": 2048     # 上下文窗口大小
        }
    }

    # Send the POST request（流式接收并拼接各片段文本）
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

result = query_model("What do Llamas eat?")  # 先验证 API 能跑通
print(result)

In [ ]:
json_file = "eval-example-data.json"  # 待评测数据：含 instruction/input/output 及各模型回复

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

In [ ]:
json_data[0]  # 看第一条的结构

In [ ]:
def format_input(entry):
    # 按 Alpaca 风格把一条样本拼成给「裁判模型」看的指令文本
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""  # 有 input 才拼这段
    # 【bug修复】原此处有一行独立的 `instruction_text + input_text`：它只是计算后把结果丢弃(死代码)，
    # 没有赋值也没有返回，纯属冗余。已删除；真正的返回在下一行。
    return instruction_text + input_text  # 指令段 + 可选输入段

In [ ]:
# 前 5 条试跑：让裁判模型按 0-100 给「model 1 response」打分，直观检查评分是否合理
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model 1 response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")

In [ ]:
from tqdm import tqdm


# 批量为某个模型(json_key 指定的回复字段)打分，返回分数列表
def generate_model_scores(json_data, json_key):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."  # 要求只输出整数，便于 int() 解析
        )
        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue  # 模型偶尔不按要求只回数字，解析失败就跳过该条（风险：会少统计几条）

    return scores

In [ ]:
from pathlib import Path

# 分别为 model 1 / model 2 打分，打印平均分并可选保存
for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model)
    print(f"\n{model}")
    print(f"Number of scores: {len(scores)} of {len(json_data)}")  # 成功解析出的分数条数
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")       # 平均分

    # Optionally save the scores（存到 scores/ 目录；注意该目录需已存在）
    save_path = Path("scores") / f"llama3-8b-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)